# 2.1 Demo - Securing Data in Unity Catalog
**Course:** Databricks Data Privacy · **Lessons:** Regulatory Compliance · Data Privacy · Key Concepts and Components · Audit Your Data · Data Isolation

Unity Catalog (UC) protects data at four levels. This demo practices each one:

| Level | UC feature | Practiced here |
|---|---|---|
| Who can reach an object | `GRANT` / `REVOKE` on catalog → schema → table/view/function | SQL shown (needs a UC metastore) |
| Which **rows** a user sees | **Row filters** (`ALTER TABLE ... SET ROW FILTER`) or dynamic views | ✅ SQL UDF + dynamic view |
| Which **values** a user sees | **Column masks** (`ALTER TABLE ... ALTER COLUMN ... SET MASK`) | ✅ SQL UDF + dynamic view |
| Knowing where PII lives | Tags, comments, Data Classification | ✅ comments, properties, PII scanner |
| Proving who did what | `system.access.audit`, table history, lineage | ✅ `DESCRIBE HISTORY` |

**Local emulation.** UC's `is_account_group_member('g')` checks the *querying* user's groups. This notebook simulates several personas with `use_identity(...)`. The policy functions you write are the same SQL as on Databricks. Locally they are `TEMPORARY` objects and are applied through a view, because open-source Delta has no `SET ROW FILTER` / `SET MASK`. The Databricks-native statements are in `databricks/sql/2.1-row-filters-column-masks.sql`.

## A. Setup: a customer table full of PII

In [ ]:
from adelab.classroom import setup
from adelab import datagen
from adelab.privacy import use_identity, scan_pii
from pyspark.sql import functions as F

DA = setup("2.1", reset=True)
datagen.land_users(f"{DA.paths.landing}/users")

(spark.read.json(f"{DA.paths.landing}/users")
      .withColumn("dob", F.to_date("dob"))
      .withColumn("signup_ts", F.to_timestamp("signup_ts"))
      .write.saveAsTable("customers"))
display(spark.table("customers").limit(5))

## B. Privileges: the UC object model

![Unity Catalog hierarchy metastore, catalog, schema, then table, volume, function and view, with USE CATALOG, USE SCHEMA and SELECT granted to the analysts group](../images/2.1-uc-securable-hierarchy.svg)

**How to read it:** each grant on the right opens one level of the path on the left. The `analysts` group needs all three to read `customers_secure`, and it has no grant on the base table `customers`. The `GRANT` statements below are exactly these arrows.

On Databricks you control access with ANSI SQL privileges on the three-level namespace. Privileges are inherited downward:
```sql
GRANT USE CATALOG ON CATALOG shop TO `analysts`;
GRANT USE SCHEMA  ON SCHEMA  shop.crm TO `analysts`;
GRANT SELECT      ON VIEW    shop.crm.customers_secure TO `analysts`;   -- the view, NOT the base table
REVOKE SELECT     ON TABLE   shop.crm.customers FROM `analysts`;
SHOW GRANTS ON TABLE shop.crm.customers;
```
Best practices: grant to **groups**, never to individuals. Give consumers access to secured views or filtered and masked tables only. Keep `MODIFY` with the pipeline's service principal.

## C. Find and label the PII first
You can't protect what you haven't found. UC **Data Classification** scans tables and suggests tags. Here is a small version: classify each column by its name and by regex matches on sampled values.

In [ ]:
display(scan_pii(spark.table("customers")))

Record the result as metadata so policies and audits can rely on it. On Databricks: `ALTER TABLE customers ALTER COLUMN email SET TAGS ('pii' = 'email')`. Locally we use column comments and table properties:

In [ ]:
%%sql
ALTER TABLE customers ALTER COLUMN email COMMENT 'PII: email';
ALTER TABLE customers ALTER COLUMN phone COMMENT 'PII: phone';
ALTER TABLE customers ALTER COLUMN ssn   COMMENT 'PII: national id (restricted)';
ALTER TABLE customers ALTER COLUMN dob   COMMENT 'PII: quasi-identifier';
ALTER TABLE customers SET TBLPROPERTIES ('data_classification' = 'confidential', 'contains_pii' = 'true');
DESCRIBE TABLE customers

## D. Personas
| Persona | Groups | Should see |
|---|---|---|
| `admin` | admins, pii_readers, eu_analysts, us_analysts | everything |
| `dpo` (data protection officer) | pii_readers, eu_analysts, us_analysts | all rows, real emails, SSN still masked |
| `eu_analyst` | eu_analysts | EU rows only, masked PII |
| `us_analyst` | us_analysts | US rows only, masked PII |
| `contractor` | none | nothing |

In [ ]:
use_identity(spark, "eu_analyst")
display(spark.sql("SELECT is_account_group_member('eu_analysts') AS eu, is_account_group_member('admins') AS admin"))

## E. Row filter function
A row filter is a boolean SQL UDF that receives one or more column values. UC calls it for every row, as the querying user.

In [ ]:
%%sql
CREATE OR REPLACE TEMPORARY FUNCTION region_filter(region STRING)
RETURNS BOOLEAN
RETURN is_account_group_member('admins')
    OR (region = 'EU' AND is_account_group_member('eu_analysts'))
    OR (region = 'US' AND is_account_group_member('us_analysts'))

## F. Column mask functions
A mask is a SQL UDF that takes the column value and returns what the current user is allowed to see. The return type must match the column type.

In [ ]:
%%sql
CREATE OR REPLACE TEMPORARY FUNCTION mask_email(email STRING)
RETURNS STRING
RETURN CASE WHEN is_account_group_member('pii_readers') THEN email
            ELSE regexp_replace(email, '^[^@]+', '*****') END;

CREATE OR REPLACE TEMPORARY FUNCTION mask_ssn(ssn STRING)
RETURNS STRING
RETURN CASE WHEN is_account_group_member('admins') THEN ssn
            ELSE concat('***-**-', right(ssn, 4)) END;

CREATE OR REPLACE TEMPORARY FUNCTION mask_phone(phone STRING)
RETURNS STRING
RETURN CASE WHEN is_account_group_member('pii_readers') THEN phone ELSE mask(phone) END;

CREATE OR REPLACE TEMPORARY FUNCTION mask_salary(salary BIGINT)
RETURNS BIGINT
RETURN CASE WHEN is_account_group_member('admins') THEN salary
            ELSE CAST(round(salary, -4) AS BIGINT) END

## G. Attach the policies
**On Databricks** you attach them to the table itself, and they apply to *every* query path (SQL, Python, BI tools):
```sql
ALTER TABLE customers SET ROW FILTER region_filter ON (region);
ALTER TABLE customers ALTER COLUMN email  SET MASK mask_email;
ALTER TABLE customers ALTER COLUMN ssn    SET MASK mask_ssn;
ALTER TABLE customers ALTER COLUMN phone  SET MASK mask_phone;
ALTER TABLE customers ALTER COLUMN salary SET MASK mask_salary;
```
**Locally** (and on Databricks before row filters existed) you use a **dynamic view** that calls the same functions. Consumers get `SELECT` on the view only.

In [ ]:
%%sql
CREATE OR REPLACE TEMPORARY VIEW customers_secure AS
SELECT user_id, first_name, last_name,
       mask_email(email)   AS email,
       mask_phone(phone)   AS phone,
       mask_ssn(ssn)       AS ssn,
       dob, gender, city, zip, region,
       mask_salary(salary) AS salary,
       signup_ts
FROM customers
WHERE region_filter(region)

## H. Same query, different users

![Four personas send the same query; the row filter and column masks, evaluated with each user's groups, return all rows, masked values, EU rows only, or zero rows](../images/2.1-row-filter-column-mask.svg)

**How to read it:** one query enters the green box once per persona. `region_filter` decides which rows survive and the `mask_*` functions decide which values are shown, both via `is_account_group_member()`. The loop below runs this for five personas (`us_analyst` is the mirror of `eu_analyst`).

In [ ]:
QUERY = "SELECT region, count(*) AS rows, first(email) AS email, first(ssn) AS ssn, first(salary) AS salary FROM customers_secure GROUP BY region ORDER BY region"

for persona in ("admin", "dpo", "eu_analyst", "us_analyst", "contractor"):
    use_identity(spark, persona)
    display(spark.sql(QUERY))

## I. Data isolation
Row filters hide data *logically*. Regulations such as GDPR data residency often require *physical* isolation instead:
- **Catalog per region or business unit**, each with its own **managed storage location** in that region's cloud account: `CREATE CATALOG shop_eu MANAGED LOCATION 'abfss://eu@...'`
- **Workspace-catalog binding**: only the EU workspace can use `shop_eu`
- Separate **service principals** per environment

![Two regional stacks: each workspace is bound to its own catalog, which stores data in a managed location in that region; the US workspace is blocked from the EU catalog](../images/2.1-data-isolation.svg)

**How to read it:** each column is one region, isolated top to bottom. The binding stops the US workspace from using `shop_eu`, and `MANAGED LOCATION` keeps EU tables in EU storage. The next cell builds a local stand-in: one schema per region, each in its own directory.

In [ ]:
for region in ("eu", "us"):
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {DA.schema_name}_{region}")
    (spark.table("customers").filter(F.col("region") == region.upper())
          .write.mode("overwrite").saveAsTable(f"{DA.schema_name}_{region}.customers"))
    loc = spark.sql(f"DESCRIBE DETAIL {DA.schema_name}_{region}.customers").first()["location"]
    print(f"{region.upper()}: {spark.table(f'{DA.schema_name}_{region}.customers').count():>4} rows at {loc}")

## J. Audit your data

![Analysts, dashboards and jobs go through Unity Catalog, which feeds the audit and lineage system tables; the Delta log of the table feeds DESCRIBE HISTORY](../images/2.1-audit-lineage.svg)

**How to read it:** the three gray boxes answer different questions. History covers writes to one table, audit covers every access (reads too), lineage covers where data flows. Only the bottom one runs locally, in the next cell.

**Table history** records every change: who, when, which operation, and with what parameters. It is the first stop when you need to show what happened to a table.

In [ ]:
%%sql
DESCRIBE HISTORY customers

On Databricks, the **audit system table** records *every access*, including reads:
```sql
SELECT event_time, user_identity.email, action_name, request_params.full_name_arg AS object
FROM system.access.audit
WHERE service_name = 'unityCatalog'
  AND action_name IN ('getTable', 'generateTemporaryTableCredential')
  AND request_params.full_name_arg = 'shop.crm.customers'
  AND event_date >= current_date() - INTERVAL 7 DAYS
ORDER BY event_time DESC;
```
**Lineage** (`system.access.table_lineage`, `column_lineage`) answers "where did this PII column flow downstream?", which you need to process deletion requests (demo 2.3).

In [ ]:
use_identity(spark, "admin")
for region in ("eu", "us"):
    spark.sql(f"DROP SCHEMA IF EXISTS {DA.schema_name}_{region} CASCADE")
DA.cleanup()